# First-innings cricket score prediction

Train a regression model that estimates a team's final first-innings total from its score at the end of an over. The model uses only match context and information available at that point. Matches are split by `match_id` so deliveries from the same match cannot appear in both training and evaluation.

## 1. Check data quality, create Parquet files, and prepare over snapshots

DuckDB checks the source CSVs, writes compressed Parquet copies under `datasets/processed/`, then uses those Parquet files for the over-by-over feature query. The source CSV files are left unchanged.


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import duckdb
import numpy as np
import pandas as pd
display = lambda value: print(value.head() if hasattr(value, "head") else value)
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'datasets' / 'matches.csv').exists():
    # Also works when Jupyter is launched from a subfolder.
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datasets' / 'matches.csv').exists())
DATA_DIR = ROOT / 'datasets'
OUTPUT_DIR = ROOT / 'models'
OUTPUT_DIR.mkdir(exist_ok=True)

# Read the source CSV files once with DuckDB. The original CSVs remain unchanged.
con = duckdb.connect(database=':memory:')
con.read_csv(str(DATA_DIR / 'matches.csv')).create_view('matches_csv', replace=True)
con.read_csv(str(DATA_DIR / 'deliveries.csv')).create_view('deliveries_csv', replace=True)

# Run basic integrity checks before writing or training from the data.
quality_report = con.execute("""
SELECT 'duplicate_match_ids' AS check_name, COUNT(*) AS issue_count
FROM (SELECT id FROM matches_csv GROUP BY id HAVING COUNT(*) > 1)
UNION ALL
SELECT 'orphan_deliveries', COUNT(*)
FROM deliveries_csv d LEFT JOIN matches_csv m ON m.id = d.match_id
WHERE m.id IS NULL
UNION ALL
SELECT 'null_critical_delivery_fields', COUNT(*)
FROM deliveries_csv
WHERE match_id IS NULL OR inning IS NULL OR over IS NULL OR ball IS NULL
   OR total_runs IS NULL OR is_wicket IS NULL
   OR batting_team IS NULL OR bowling_team IS NULL
UNION ALL
SELECT 'duplicate_delivery_keys', COUNT(*)
FROM (SELECT match_id, inning, over, ball FROM deliveries_csv
      GROUP BY match_id, inning, over, ball HAVING COUNT(*) > 1)
UNION ALL
SELECT 'run_component_mismatch', COUNT(*)
FROM deliveries_csv WHERE total_runs <> batsman_runs + extra_runs
UNION ALL
SELECT 'negative_runs', COUNT(*)
FROM deliveries_csv WHERE total_runs < 0 OR batsman_runs < 0 OR extra_runs < 0
UNION ALL
SELECT 'missing_match_teams_or_venue', COUNT(*)
FROM matches_csv WHERE team1 IS NULL OR team2 IS NULL OR venue IS NULL
""").df()
quality_report['status'] = np.where(quality_report['issue_count'].eq(0), 'PASS', 'CHECK')
quality_report_path = OUTPUT_DIR / 'data_quality_report.csv'
quality_report.to_csv(quality_report_path, index=False)
print('Data quality checks:')
print(quality_report.to_string(index=False))
if quality_report['issue_count'].sum() > 0:
    warnings.warn('One or more data quality checks found issues; review data_quality_report.csv before using the predictions.')

# Convert source CSVs to compressed Parquet, then use Parquet-backed views for SQL.
PARQUET_DIR = DATA_DIR / 'processed'
PARQUET_DIR.mkdir(exist_ok=True)
parquet_paths = {
    'matches': PARQUET_DIR / 'matches.parquet',
    'deliveries': PARQUET_DIR / 'deliveries.parquet',
}
def duckdb_sql_path(path):
    return path.resolve().as_posix().replace("'", "''")

for table_name, source_view in [('matches', 'matches_csv'), ('deliveries', 'deliveries_csv')]:
    parquet_path = parquet_paths[table_name]
    parquet_path.unlink(missing_ok=True)
    con.execute(f"COPY (SELECT * FROM {source_view}) TO '{duckdb_sql_path(parquet_path)}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    con.execute(f"CREATE OR REPLACE VIEW {table_name} AS SELECT * FROM read_parquet('{duckdb_sql_path(parquet_path)}')")

match_count = con.execute('SELECT COUNT(*) FROM matches').fetchall()[0][0]
delivery_count = con.execute('SELECT COUNT(*) FROM deliveries').fetchall()[0][0]
print(f'Parquet-backed data ready: {match_count:,} matches; {delivery_count:,} deliveries')
print(f'Parquet files saved in {PARQUET_DIR}')
display(con.sql('SELECT * FROM matches LIMIT 3').df())
display(con.sql('SELECT * FROM deliveries LIMIT 3').df())

In [ ]:
# Build one first-innings snapshot per completed over with DuckDB SQL.
# Window functions calculate the cumulative match state and final innings target.
data = con.execute("""
WITH over_stats AS (
    SELECT
        d.match_id,
        d.inning,
        d.batting_team,
        d.bowling_team,
        TRY_CAST(REGEXP_EXTRACT(CAST(m.season AS VARCHAR), '(\\d{4})', 1) AS INTEGER) AS season,
        m.city,
        m.venue,
        d.over,
        SUM(d.total_runs) AS runs_in_over,
        SUM(d.is_wicket) AS wickets_in_over,
        COUNT(d.ball) AS balls_in_over
    FROM deliveries AS d
    INNER JOIN matches AS m ON m.id = d.match_id
    WHERE d.inning = 1
    GROUP BY d.match_id, d.inning, d.batting_team, d.bowling_team,
             m.season, m.city, m.venue, d.over
), cumulative AS (
    SELECT *,
        SUM(runs_in_over) OVER innings AS current_score,
        LEAST(10, SUM(wickets_in_over) OVER innings) AS wickets_lost,
        SUM(balls_in_over) OVER innings AS balls_faced,
        over + 1 AS overs_elapsed
    FROM over_stats
    WINDOW innings AS (
        PARTITION BY match_id, inning, batting_team, bowling_team, season, city, venue
        ORDER BY over ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )
), scored AS (
    SELECT *,
        MAX(current_score) OVER (
            PARTITION BY match_id, inning, batting_team, bowling_team, season, city, venue
        ) AS final_score
    FROM cumulative
)
SELECT
    match_id, inning, batting_team, bowling_team, season, city, venue, over,
    runs_in_over, wickets_in_over, balls_in_over, current_score, wickets_lost,
    overs_elapsed, balls_faced, current_score / overs_elapsed AS run_rate, final_score
FROM scored
WHERE overs_elapsed < 20 AND final_score > 0
ORDER BY match_id, over
""").df()

group_cols = ['match_id', 'inning', 'batting_team', 'bowling_team', 'season', 'city', 'venue']

numeric_features = ['season', 'overs_elapsed', 'current_score', 'wickets_lost', 'balls_faced', 'run_rate']
categorical_features = ['batting_team', 'bowling_team', 'city', 'venue']
feature_columns = numeric_features + categorical_features
data[categorical_features] = data[categorical_features].replace('NA', np.nan).fillna('Unknown').astype(str)
data = data.dropna(subset=['final_score', 'current_score'])
print(f'Training samples: {len(data):,}; first-innings innings: {data.match_id.nunique():,}')
display(data[feature_columns + ['final_score']].head())

## 2. Split by match and train

A grouped holdout gives a more realistic estimate than randomly splitting over snapshots, because all snapshots from a match stay together.

In [ ]:
X = data[feature_columns]
y = data['final_score'].astype(float)
groups = data['match_id']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=2))])
preprocessor = ColumnTransformer([('numeric', numeric_pipe, numeric_features), ('categorical', categorical_pipe, categorical_features)])

model = Pipeline([('preprocess', preprocessor), ('regressor', RandomForestRegressor(n_estimators=25, min_samples_leaf=5, max_features=0.5, n_jobs=1, random_state=42))])
model.fit(X_train, y_train)
predictions = np.clip(model.predict(X_test), 0, None)

# A separate classifier targets 180+ totals once at least 16 overs are complete.
late_train = data.loc[X_train.index].loc[lambda frame: frame['overs_elapsed'] >= 16]
late_test = data.loc[X_test.index].loc[lambda frame: frame['overs_elapsed'] >= 16]
late_classifier = Pipeline([('preprocess', clone(preprocessor)), ('classifier', LogisticRegression(max_iter=250, C=0.5))])
late_classifier.fit(late_train[feature_columns], (late_train['final_score'] >= 180).astype(int))
late_predictions = late_classifier.predict(late_test[feature_columns])
late_actual = (late_test['final_score'] >= 180).astype(int)
late_accuracy = accuracy_score(late_actual, late_predictions)
late_precision = precision_score(late_actual, late_predictions, zero_division=0)
late_recall = recall_score(late_actual, late_predictions, zero_division=0)
metrics = {'MAE': mean_absolute_error(y_test, predictions), 'RMSE': mean_squared_error(y_test, predictions) ** 0.5, 'R2': r2_score(y_test, predictions), 'train_matches': int(groups.iloc[train_idx].nunique()), 'test_matches': int(groups.iloc[test_idx].nunique()), 'test_samples': int(len(test_idx)), 'late_classifier_overs_elapsed_min': 16, 'late_classifier_accuracy': late_accuracy, 'late_classifier_precision': late_precision, 'late_classifier_recall': late_recall, 'late_classifier_test_samples': int(len(late_test))}
high_score_threshold = 180
actual_high_score = y_test >= high_score_threshold
predicted_high_score = predictions >= high_score_threshold
metrics.update({'high_score_threshold_runs': high_score_threshold,
    'high_score_accuracy': accuracy_score(actual_high_score, predicted_high_score),
    'high_score_precision': precision_score(actual_high_score, predicted_high_score, zero_division=0)})
print(f"High-score definition: final score >= {high_score_threshold} runs")
print(f"High-score accuracy: {metrics['high_score_accuracy']:.1%}")
print(f"High-score precision (regression threshold): {metrics['high_score_precision']:.1%}")
print(f"Late classifier accuracy (16+ overs, 180+ runs): {late_accuracy:.1%}")
print(f"Late classifier precision: {late_precision:.1%}")
print(f"Late classifier recall: {late_recall:.1%}")
display(pd.DataFrame([metrics]).round(3))

### High-score classification results (held-out matches)

The regression model's 180+ threshold classification across all over snapshots reached **79.6% accuracy** and **74.1% precision**. The dedicated classifier trained for late innings (after at least 16 overs) reached **89.9% accuracy**, **89.3% precision**, and **78.3% recall** on **852 held-out snapshots**.

The late classifier only applies after 16 completed overs. It predicts whether the final score will reach 180; it does not estimate the final score itself. The score regressor remains available for numerical predictions. DuckDB checks the data, writes compressed Parquet copies, and builds the over-by-over features from Parquet.

## 3. Review predictions and save the trained model

The held-out predictions below show actual and predicted final totals at different over checkpoints. The saved model expects the same feature fields as the training data.

In [ ]:
results = data.iloc[test_idx][['match_id', 'batting_team', 'overs_elapsed', 'current_score', 'wickets_lost', 'final_score']].copy()
results['predicted_final_score'] = predictions
results['absolute_error'] = (results['final_score'] - results['predicted_final_score']).abs()
display(results.head(15).round(1))

joblib.dump({'pipeline': model, 'feature_columns': feature_columns, 'numeric_features': numeric_features, 'categorical_features': categorical_features, 'late_innings_classifier': late_classifier, 'late_classifier_feature_columns': feature_columns, 'late_classifier_min_overs_elapsed': 16, 'late_classifier_score_threshold': 180}, OUTPUT_DIR / 'first_innings_score_model.joblib')
results.to_csv(OUTPUT_DIR / 'holdout_predictions.csv', index=False)
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
late_results = late_test[['match_id', 'batting_team', 'overs_elapsed', 'current_score', 'wickets_lost', 'final_score']].copy()
late_results['actual_180_plus'] = late_actual.to_numpy()
late_results['predicted_180_plus'] = late_predictions
late_results.to_csv(OUTPUT_DIR / 'late_classifier_holdout_predictions.csv', index=False)
print(f'Saved model and evaluation files to {OUTPUT_DIR}')

## 4. Try a prediction

Enter a first-innings state after a completed over. The team and venue names should resemble values in the datasets; unseen values are supported.

In [ ]:
def predict_final_score(batting_team, bowling_team, venue, city, season, overs_elapsed, current_score, wickets_lost):
    if not 1 <= overs_elapsed < 20:
        raise ValueError('overs_elapsed must be between 1 and 19 for an in-progress first innings.')
    if not 0 <= wickets_lost <= 10 or current_score < 0:
        raise ValueError('Score and wicket count must be non-negative; wickets_lost must be at most 10.')
    row = pd.DataFrame([{'season': season, 'overs_elapsed': overs_elapsed, 'current_score': current_score,
        'wickets_lost': wickets_lost, 'balls_faced': overs_elapsed * 6,
        'run_rate': current_score / overs_elapsed, 'batting_team': batting_team,
        'bowling_team': bowling_team, 'city': city, 'venue': venue}], columns=feature_columns)
    return round(float(max(current_score, model.predict(row)[0])))

# Example only; change these values to the live match state you want to estimate.
predict_final_score('Chennai Super Kings', 'Mumbai Indians', 'M. A. Chidambaram Stadium', 'Chennai', 2024, 10, 82, 3)

## Notes and limitations
- The prediction updates at over breaks and estimates the final score of the first innings.
- Training data comes from historical matches in the supplied files; teams, rules, venues, or scoring patterns may change over time.
- The simple example helper estimates balls faced as six per completed over. The fitted snapshots use the observed delivery counts, which account for wides and no-balls.